# 🤖 Machine Learning
**Cours couverts :** Understanding Machine Learning, Supervised Learning with scikit-learn, Machine Learning with Tree-Based Models in Python, Unsupervised Learning in Python, Cluster Analysis in Python, Dimensionality Reduction in Python

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification, make_regression, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    mean_squared_error, r2_score, mean_absolute_error
)
from sklearn.pipeline import Pipeline

np.random.seed(42)
print("scikit-learn importé avec succès")

---
## PARTIE 1 — Apprentissage Supervisé

**Principe :** On apprend à partir d'exemples étiquetés (X, y). L'objectif est de prédire y pour de nouveaux X.

### 1.1 Workflow général scikit-learn

In [ ]:
# Workflow scikit-learn standard
# 1. Charger les données
iris = load_iris()
X, y = iris.data, iris.target

# 2. Diviser en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {iris.target_names}")

# 3. Prétraitement (normalisation)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit ET transform sur train
X_test_sc  = scaler.transform(X_test)       # SEULEMENT transform sur test !

# 4. Choisir et entraîner un modèle
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_sc, y_train)

# 5. Évaluer
y_pred = model.predict(X_test_sc)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nRapport de classification:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### 1.2 Algorithmes de classification

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

classifiers = {
    "Régression Logistique": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", C=1.0),
    "Naive Bayes": GaussianNB()
}

print("Comparaison des modèles (Accuracy sur test) :")
print("-" * 45)
for nom, clf in classifiers.items():
    clf.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_sc))
    cv_scores = cross_val_score(clf, X, y, cv=5)
    print(f"{nom:<25} Test: {acc:.3f} | CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# Matrice de confusion
best_clf = LogisticRegression(max_iter=1000)
best_clf.fit(X_train_sc, y_train)
y_pred_best = best_clf.predict(X_test_sc)

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
ax.set_ylabel("Valeur réelle")
ax.set_xlabel("Valeur prédite")
ax.set_title("Matrice de confusion")
plt.tight_layout()
plt.show()

### 1.3 Régression

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

X_reg, y_reg = make_regression(n_samples=200, n_features=10, noise=20, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

regressors = {
    "Régression Linéaire": LinearRegression(),
    "Ridge (L2)": Ridge(alpha=1.0),
    "Lasso (L1)": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5)
}

print("Modèles de régression (RMSE et R²) :")
print("-" * 55)
for nom, reg in regressors.items():
    reg.fit(X_train_r, y_train_r)
    y_pred_r = reg.predict(X_test_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    r2 = r2_score(y_test_r, y_pred_r)
    print(f"{nom:<22} RMSE: {rmse:.2f} | R²: {r2:.3f}")

print("\nRégularisation:")
print("  Ridge (L2) : réduit tous les coefficients → bonne quand features corrélées")
print("  Lasso (L1) : met des coefficients à 0 → sélection de features automatique")
print("  ElasticNet : combinaison Ridge + Lasso")

---
## PARTIE 2 — Modèles à base d'arbres

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# Données de classification binaire
X_cls, y_cls = make_classification(n_samples=1000, n_features=10, n_informative=5,
                                    n_redundant=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Arbre de décision seul
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(15, 5))
plot_tree(dt, max_depth=2, filled=True, feature_names=[f"f{i}" for i in range(10)],
          class_names=["0", "1"], ax=ax, fontsize=8)
ax.set_title("Arbre de décision (profondeur max affichée = 2)")
plt.tight_layout()
plt.show()
print(f"Accuracy arbre seul : {accuracy_score(y_te, dt.predict(X_te)):.3f}")

In [ ]:
# Ensemble methods
try:
    ensembles = {
        "Arbre seul (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", verbosity=0)
    }
except ImportError:
    ensembles = {
        "Arbre seul (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42)
    }
    print("XGBoost non disponible, continuer sans")

resultats = {}
print("Comparaison des modèles à base d'arbres :")
print("-" * 55)
for nom, clf in ensembles.items():
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    resultats[nom] = acc
    print(f"{nom:<30} Accuracy: {acc:.3f}")

print("\nConcepts clés:")
print("  Bagging (RF)    : arbres indépendants, bootstrap, moyenne → réduit la variance")
print("  Boosting (GB/XGB): arbres séquentiels, corrige les erreurs → réduit le biais")

In [ ]:
# Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)

importances = pd.Series(rf.feature_importances_,
                         index=[f"feature_{i}" for i in range(10)]).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Importance des features (Random Forest)")
ax.set_xlabel("Feature")
ax.set_ylabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Overfitting vs Underfitting : courbes de validation
profondeurs = range(1, 20)
train_scores = []
test_scores = []

for d in profondeurs:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_scores.append(accuracy_score(y_tr, dt.predict(X_tr)))
    test_scores.append(accuracy_score(y_te, dt.predict(X_te)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(profondeurs, train_scores, label="Train", color="steelblue", marker="o")
ax.plot(profondeurs, test_scores, label="Test", color="coral", marker="o")
ax.set_xlabel("Profondeur maximale de l'arbre")
ax.set_ylabel("Accuracy")
ax.set_title("Overfitting : la profondeur augmente le score train mais peut dégrader le test")
ax.legend()
ax.axvspan(1, 3, alpha=0.1, color="orange", label="Underfitting")
ax.axvspan(12, 19, alpha=0.1, color="red", label="Overfitting")
ax.legend()
plt.tight_layout()
plt.show()

### Pipelines et GridSearchCV

In [ ]:
# Pipeline : chaîner prétraitement + modèle
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=42))
])

# GridSearchCV : recherche des meilleurs hyperparamètres
param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring="accuracy",
                            n_jobs=-1, verbose=0)
grid_search.fit(X_tr, y_tr)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur score CV  : {grid_search.best_score_:.3f}")
print(f"Score test         : {accuracy_score(y_te, grid_search.predict(X_te)):.3f}")

---
## PARTIE 3 — Apprentissage Non Supervisé

### 3.1 Clustering

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Générer des données avec des clusters naturels
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# --- K-Means ---
# Trouver le bon k avec la méthode du coude
inertias = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, marker="o", color="steelblue")
axes[0].set_title("Méthode du coude (Elbow Method)")
axes[0].set_xlabel("Nombre de clusters (k)")
axes[0].set_ylabel("Inertie")

axes[1].plot(K_range, silhouettes, marker="o", color="coral")
axes[1].set_title("Score de silhouette (plus élevé = meilleur)")
axes[1].set_xlabel("Nombre de clusters (k)")
axes[1].set_ylabel("Score silhouette")

plt.tight_layout()
plt.show()
print(f"k optimal (silhouette) : {K_range[silhouettes.index(max(silhouettes))]}")

In [ ]:
# Visualiser les clusters K-Means, Agglomeratif et DBSCAN
algorithms = [
    ("K-Means (k=4)", KMeans(n_clusters=4, random_state=42, n_init=10)),
    ("Agglomeratif (k=4)", AgglomerativeClustering(n_clusters=4)),
    ("DBSCAN", DBSCAN(eps=0.8, min_samples=5))
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nom, alg) in zip(axes, algorithms):
    labels = alg.fit_predict(X_blobs)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="tab10", alpha=0.7)
    if hasattr(alg, "cluster_centers_"):
        ax.scatter(alg.cluster_centers_[:, 0], alg.cluster_centers_[:, 1],
                   marker="*", s=200, color="black", zorder=5)
    ax.set_title(f"{nom}\n{n_clusters} clusters trouvés")

plt.suptitle("Comparaison algorithmes de clustering")
plt.tight_layout()
plt.show()

print("\nDBSCAN : -1 = points de bruit (ne font partie d'aucun cluster)")
print("DBSCAN ne nécessite pas de spécifier k, mais eps et min_samples")

### 3.2 Réduction de dimensionnalité

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# --- PCA (Analyse en Composantes Principales) ---
# Données : iris avec 4 features → réduire à 2D pour visualiser
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Normaliser AVANT PCA
scaler = StandardScaler()
X_iris_sc = scaler.fit_transform(X_iris)

# PCA complète pour voir la variance expliquée
pca_full = PCA()
pca_full.fit(X_iris_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Variance expliquée par composante
axes[0].bar(range(1, 5), pca_full.explained_variance_ratio_, color="steelblue", edgecolor="white")
axes[0].plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_),
             color="red", marker="o", label="Cumulatif")
axes[0].axhline(0.95, color="orange", linestyle="--", label="95%")
axes[0].set_title("Variance expliquée par composante")
axes[0].set_xlabel("Composante principale")
axes[0].set_ylabel("Proportion de variance")
axes[0].legend()

# Projection 2D
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_iris_sc)

for classe, nom in enumerate(iris.target_names):
    mask = y_iris == classe
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=nom, alpha=0.7)
axes[1].set_title(f"PCA 2D\n({pca_2d.explained_variance_ratio_.sum():.1%} de variance expliquée)")
axes[1].set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- t-SNE (non-linéaire, meilleur pour visualisation) ---
# Attention : t-SNE est UNIQUEMENT pour la visualisation, pas pour le prétraitement ML
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_iris_sc)

fig, ax = plt.subplots(figsize=(7, 5))
for classe, nom in enumerate(iris.target_names):
    mask = y_iris == classe
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=nom, alpha=0.7)
ax.set_title("t-SNE 2D (meilleure séparation des clusters)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nDifférences PCA vs t-SNE:")
print("  PCA    : linéaire, préserve la variance globale, interprétable, peut servir de prétraitement")
print("  t-SNE  : non-linéaire, préserve la structure locale, UNIQUEMENT pour visualisation")

### 3.3 Dendrogramme (Clustering Hiérarchique)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Dendrogramme sur un sous-ensemble de données
X_small = X_blobs[:50]

linked = linkage(X_small, method="ward")

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(linked, ax=ax, leaf_rotation=90, leaf_font_size=8, truncate_mode="level", p=4)
ax.set_title("Dendrogramme (Ward linkage)")
ax.set_xlabel("Index des points")
ax.set_ylabel("Distance")
ax.axhline(y=5, color="red", linestyle="--", label="Coupe → 4 clusters")
ax.legend()
plt.tight_layout()
plt.show()

print("Lire un dendrogramme : couper horizontalement donne le nombre de clusters.")
print("Plus les branches sont hautes, plus les clusters sont différents.")

## Résumé : Choisir son algorithme ML

In [ ]:
guide = """
CLASSIFICATION (prédire une catégorie) :
  • Petits données, baseline            → Régression Logistique, Naive Bayes
  • Bonnes performances générales       → Random Forest, Gradient Boosting
  • Meilleures performances (compétition) → XGBoost, LightGBM
  • Données texte/images                → SVM (noyau RBF), Deep Learning

RÉGRESSION (prédire un nombre) :
  • Baseline                            → Régression Linéaire
  • Features corrélées / multicolinéarité → Ridge
  • Sélection de features automatique   → Lasso
  • Bonnes performances                 → Random Forest Regressor, XGBoost

CLUSTERING (grouper des données) :
  • k connu, clusters sphériques        → K-Means
  • Pas de k fixe, clusters de formes variées → DBSCAN
  • Comprendre la hiérarchie             → Clustering Agglomératif + Dendrogramme

RÉDUCTION DE DIMENSIONNALITÉ :
  • Prétraitement avant ML              → PCA
  • Visualisation uniquement            → t-SNE, UMAP

MÉTRIQUES :
  • Classification équilibrée           → Accuracy
  • Classes déséquilibrées              → F1-score, Precision, Recall, AUC-ROC
  • Régression                          → RMSE (pénalise les grandes erreurs), MAE (robuste), R²
  • Clustering                          → Silhouette score, Inertie
"""
print(guide)